In [ ]:
import json
from collections import defaultdict
import pandas as pd


# -----------------------------
# 1. Load dataset
# -----------------------------
with open("./support_files/updated_traits.json", "r") as f:
    data = json.load(f)

# -----------------------------
# 2. Define invasive vs. non-invasive
# -----------------------------
invasive_species = {"lythrum salicaria", "lythrum virgatum", "lythrum hyssopifolia"}

invasive = {}
non_invasive = {}

for species, traits in data.items():
    if species in invasive_species:
        invasive[species] = traits
    else:
        non_invasive[species] = traits

# -----------------------------
# 3. Collect features
# -----------------------------
def collect_features(species_dict):
    features = defaultdict(set)
    for traits in species_dict.values():
        for group, subfeatures in traits.items():
            for f in subfeatures:
                features[group].add(f)
    return features

invasive_features = collect_features(invasive)
non_invasive_features = collect_features(non_invasive)

# -----------------------------
# 4. Identify unique + common features
# -----------------------------
unique_invasive = {}
unique_non_invasive = {}
common_features = {}

for group in ["leaf", "flower", "stem"]:
    inv = invasive_features[group]
    noninv = non_invasive_features[group]
    unique_invasive[group] = inv - noninv
    unique_non_invasive[group] = noninv - inv
    common_features[group] = inv & noninv

# -----------------------------
# 5. Build a DataFrame for visualization
# -----------------------------
rows = []
for group in ["leaf", "flower", "stem"]:
    for f in sorted(unique_invasive[group]):
        rows.append([group, f, "Invasive only"])
    for f in sorted(unique_non_invasive[group]):
        rows.append([group, f, "Non-invasive only"])
    for f in sorted(common_features[group]):
        rows.append([group, f, "Common to both"])

df = pd.DataFrame(rows, columns=["Feature group", "Feature", "Category"])

# -----------------------------
# 6. Print results
# -----------------------------
print("\n=== Unique invasive features ===")
for group, feats in unique_invasive.items():
    print(f"{group}: {sorted(feats)}")

print("\n=== Unique non-invasive features ===")
for group, feats in unique_non_invasive.items():
    print(f"{group}: {sorted(feats)}")

print("\n=== Common features (both invasive and non-invasive) ===")
for group, feats in common_features.items():
    print(f"{group}: {sorted(feats)}")

print("\n=== Table of all features by category ===")
print(df.to_string(index=False))


=== Unique invasive features ===
leaf: ['narrower at the base']
flower: ['inflorescence spikelike to raceme', 'stamens 10-14']
stem: ['erect to weakly erect']

=== Unique non-invasive features ===
leaf: ['a discernible gap between the stem and the base of the blade', 'attenuate at the base', 'broader at the base', 'cordate at the base', 'linear', 'needle-like', 'obovate', 'obtuse to truncate at the base', 'petiolated', 'sessile or subsessile', 'subsessile']
flower: ['calyx with teeth of the same length', 'floral tube campanulate', 'floral tube narrowly cylindrical without red dots', 'floral tube obconic', 'floral tube red dotted', 'flowers solitary', 'flowers solitary in leaf axils', 'one or few flowers in the axil of leaves', 'petals 4', 'petals lavender', 'petals lilac to pink', 'petals minute', 'petals pale purple or whitish', 'petals pink to purple with a darker midrib', 'petals purple with a darker midrib', 'petals reddish', 'petals white', 'petals white to pink', 'stamens 10-12'

In [ ]:
#!/usr/bin/env python3
"""
Lythrum Trait Analysis
----------------------

This script analyzes traits of invasive vs. non-invasive Lythrum species.

- Input: updated_traits.json (same directory)
- Invasive species: Lythrum salicaria, Lythrum virgatum, Lythrum hyssopifolia
- Outputs:
    * Unique features (invasive-only, non-invasive-only, common)
    * Unique feature pairs (invasive-only, non-invasive-only, common)

"""

import json
from collections import defaultdict, Counter
import itertools
import pandas as pd


# -----------------------------
# 1. Load dataset
# -----------------------------
with open("./support_files/updated_traits.json", "r") as f:
    data = json.load(f)

# -----------------------------
# 2. Define invasive vs. non-invasive
# -----------------------------
invasive_species = {"lythrum salicaria", "lythrum virgatum", "lythrum hyssopifolia"}

invasive = {}
non_invasive = {}

for species, traits in data.items():
    if species in invasive_species:
        invasive[species] = traits
    else:
        non_invasive[species] = traits

# -----------------------------
# 3. Collect features
# -----------------------------
def collect_features(species_dict):
    features = defaultdict(set)
    for traits in species_dict.values():
        for group, subfeatures in traits.items():
            for f in subfeatures:
                features[group].add(f)
    return features

invasive_features = collect_features(invasive)
non_invasive_features = collect_features(non_invasive)

# -----------------------------
# 4. Identify unique + common single features
# -----------------------------
unique_invasive = {}
unique_non_invasive = {}
common_features = {}

for group in ["leaf", "flower", "stem"]:
    inv = invasive_features[group]
    noninv = non_invasive_features[group]
    unique_invasive[group] = inv - noninv
    unique_non_invasive[group] = noninv - inv
    common_features[group] = inv & noninv

# -----------------------------
# 5. Collect feature pairs per species, with counts
# -----------------------------
def collect_pairs_with_counts(species_dict):
    """Return Counter of all feature pairs across given species."""
    counter = Counter()
    for traits in species_dict.values():
        # Flatten all features across groups
        all_feats = []
        for group, subfeatures in traits.items():
            all_feats.extend(subfeatures)
        # Generate all unordered pairs
        for a, b in itertools.combinations(sorted(set(all_feats)), 2):
            counter[(a, b)] += 1
    return counter

invasive_pairs_counts = collect_pairs_with_counts(invasive)
non_invasive_pairs_counts = collect_pairs_with_counts(non_invasive)

# -----------------------------
# 6. Identify unique + common pairs with counts
# -----------------------------
invasive_pairs = set(invasive_pairs_counts.keys())
non_invasive_pairs = set(non_invasive_pairs_counts.keys())

unique_invasive_pairs = invasive_pairs - non_invasive_pairs
unique_non_invasive_pairs = non_invasive_pairs - invasive_pairs
common_pairs = invasive_pairs & non_invasive_pairs

# Filter counts for each category
unique_invasive_counts = {p: invasive_pairs_counts[p] for p in unique_invasive_pairs}
unique_non_invasive_counts = {p: non_invasive_pairs_counts[p] for p in unique_non_invasive_pairs}
common_counts = {p: invasive_pairs_counts[p] + non_invasive_pairs_counts[p] for p in common_pairs}

# -----------------------------
# 7. Build DataFrames for output
# -----------------------------
def make_df(pair_dict, category):
    rows = []
    for (a, b), count in sorted(pair_dict.items(), key=lambda x: -x[1]):
        rows.append([a, b, count, category])
    return pd.DataFrame(rows, columns=["Feature A", "Feature B", "Count", "Category"])

df_invasive_pairs = make_df(unique_invasive_counts, "Invasive only")
df_non_invasive_pairs = make_df(unique_non_invasive_counts, "Non-invasive only")
df_common_pairs = make_df(common_counts, "Common to both")

# -----------------------------
# 8. Print results
# -----------------------------
print("\n=== Unique invasive features ===")
for group, feats in unique_invasive.items():
    print(f"{group}: {sorted(feats)}")

print("\n=== Unique non-invasive features ===")
for group, feats in unique_non_invasive.items():
    print(f"{group}: {sorted(feats)}")

print("\n=== Common features (both groups) ===")
for group, feats in common_features.items():
    print(f"{group}: {sorted(feats)}")

print("\n=== Top 25 invasive-only pairs ===")
print(df_invasive_pairs.head(25).to_string(index=False))

print("\n=== Top 25 non-invasive-only pairs ===")
print(df_non_invasive_pairs.head(25).to_string(index=False))

print("\n=== Top 25 common pairs ===")
print(df_common_pairs.head(25).to_string(index=False))


=== Unique invasive features ===
leaf: ['narrower at the base']
flower: ['inflorescence spikelike to raceme', 'stamens 10-14']
stem: ['erect to weakly erect']

=== Unique non-invasive features ===
leaf: ['a discernible gap between the stem and the base of the blade', 'attenuate at the base', 'broader at the base', 'cordate at the base', 'linear', 'needle-like', 'obovate', 'obtuse to truncate at the base', 'petiolated', 'sessile or subsessile', 'subsessile']
flower: ['calyx with teeth of the same length', 'floral tube campanulate', 'floral tube narrowly cylindrical without red dots', 'floral tube obconic', 'floral tube red dotted', 'flowers solitary', 'flowers solitary in leaf axils', 'one or few flowers in the axil of leaves', 'petals 4', 'petals lavender', 'petals lilac to pink', 'petals minute', 'petals pale purple or whitish', 'petals pink to purple with a darker midrib', 'petals purple with a darker midrib', 'petals reddish', 'petals white', 'petals white to pink', 'stamens 10-12'